## STUDENT EXERCISE: Information Retrieval with Cosine Similarity + Precision & Recall

### Goal

Load a labeled text dataset

Turn documents into vectors (CountVectorizer + TF-IDF)

Use a query to retrieve similar documents (cosine similarity)

Label retrieved documents as correct/incorrect

Build a confusion matrix

Calculate precision, recall, and F1-score

Dataset: `20 Newsgroups` (already labeled)

This dataset is built into scikit-learn.

We will use only 2 categories to keep it simple.

# Step 1 - Import Libraries

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd

Step 2 - Load a Small Labeled Dataset

We choose only 2 categories so precision/recall are easy to see.

In [ ]:
'''
categories = ['rec.sport.hockey', 'sci.space']

data = fetch_20newsgroups(subset='train', categories=categories)
docs = data.data                      # list of documents
labels = data.target                  # 0 or 1
label_names = data.target_names
'''

In [ ]:
data = {
    "text": [
        "The spaceship launched successfully and entered orbit.",
        "Astronauts conduct research in the space station.",
        "New discoveries in astronomy reveal distant galaxies.",
        "The hockey team won the championship last night.",
        "The player scored three goals in the hockey match.",
        "Hockey fans celebrated the victory downtown."
    ],
    "label": [
        "space",
        "space",
        "space",
        "hockey",
        "hockey",
        "hockey"
    ]
}

df = pd.DataFrame(data)
docs = df["text"].tolist()
labels = df["label"].tolist()
label_names = df["label"].unique().tolist()

print(df)

Step 3 - Vectorize the Documents (CountVectorizer)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

count_vec = CountVectorizer(stop_words='english')
X_counts = count_vec.fit_transform(docs)

In [ ]:
words = count_vec.get_feature_names_out()
print(words)

# Bag-of-Words model
(row index, column index)   word_count

It will look with something like this

(1, 0)    1

(1, 4)    1

(1, 19)   1

*This means:*

* Document 1

* Contains words at vocabulary positions 0, 4, and 19

* Each word appears once.

How does Sparse Matrix works:

If you have 6 documents with 1000 unique words.

This would create a 6 × 1000 matrix.

In [ ]:
# Bag-of-Words model
print(X_counts)
# A normal table dataframe
#df_counts = pd.DataFrame(X_counts.toarray(), columns=count_vec.get_feature_names_out())
#df_counts

Step 3.1 - TF-IDF

In [ ]:
tfidf = TfidfTransformer()
X_tfidf = tfidf.fit_transform(X_counts)

In [ ]:
print(X_tfidf)

Step 4 - Choose a Query Document

In [ ]:
query_index = 0  # choose first doc
query = docs[query_index]
query_label = labels[query_index]

print("Query:", query)
print("Label:", query_label)

Step 5 - Compute Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query_vec = tfidf.transform(count_vec.transform([query]))
sims = cosine_similarity(query_vec, X_tfidf).flatten()

In [ ]:
print(sims)

Step 6 - Top-k Retrieval

In [ ]:
k = 3
topk_indices = sims.argsort()[-k:][::-1]

In [ ]:
print(topk_indices)
for rank, idx in enumerate(topk_indices, start=1):
    print(f"{rank}. doc_id={idx} | {docs[idx]}")

In [ ]:
for rank, idx in enumerate(topk_indices, start=1):
    print(f"{rank}. doc_id={idx} | similarity={sims[idx]:.4f} | {docs[idx]} | {labels[idx]}")

Step 7 - Build True/Predicted Relevance Arrays

In [ ]:
print(labels[0])
print(query_label)

In [ ]:
y_true = [1 if labels[i] == query_label else 0 for i in topk_indices]
y_pred = [1] * k

In [ ]:
print(y_true)

Step 8 - Confusion Matrix & Metrics

|                         | Predicted 0 (Not Relevant) | Predicted 1 (Retrieved/Relevant) |
| ----------------------- | -------------------------- | -------------------------------- |
| Actual 0 (Not Relevant) | TN                         | FP                               |
| Actual 1 (Relevant)     | FN                         | TP                               |


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nMetrics:\n", classification_report(y_true, y_pred, target_names=["Not Relevant", "Relevant"]))

### Treshold based:

Now using the code above, instead of top 3 documents, compute similarity based on a *treshold*:



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query_vec = tfidf.transform(count_vec.transform([query]))
sims = cosine_similarity(query_vec, X_tfidf).flatten()

sims = similarity of query to all documents

Each value ∈ [0,1]

threshold = 0.2

Any document with similarity ≥ threshold will be retrieved

Documents with similarity < threshold are not retrieved

In [ ]:
threshold = 0.2  # choose a value between 0 and 1

retrieved_indices = np.where(sims >= threshold)[0] # add [0] to get a clean array of indices.
print("Retrieved document indices:", retrieved_indices)

To check for relevance:

Build y_true and y_pred

True relevance:

* 1 = actually relevant

* 0 = not relevant

In [ ]:
# 1 = predicted relevant (retrieved), 0 = not retrieved
y_pred_full = [1 if s >= threshold else 0 for s in sims]
y_true_full = [1 if label == query_label else 0 for label in labels]

In [ ]:
print(y_pred_full)
print(y_true_full)

In [ ]:
for idx in np.where(np.array(y_pred) == 1)[0]:
    print(f"Doc {idx} | similarity={sims[idx]:.4f} | {docs[idx]}")

Confusion Matrix

|                         | Predicted 0 (Not Relevant) | Predicted 1 (Retrieved/Relevant) |
| ----------------------- | -------------------------- | -------------------------------- |
| Actual 0 (Not Relevant) | TN                         | FP                               |
| Actual 1 (Relevant)     | FN                         | TP                               |

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


print("Confusion Matrix:\n", confusion_matrix(y_true_full, y_pred_full))
print(classification_report(y_true_full, y_pred_full, target_names=["Not Relevant", "Relevant"]))

# PRACTICE:  Iris Dataset 

In [122]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
X = iris.data       # features
y = iris.target     # labels: 0,1,2
label_names = iris.target_names

print("Feature shape:", X.shape)
print("Labels:", y[:10])
print(label_names)

Feature shape: (150, 4)
Labels: [0 0 0 0 0 0 0 0 0 0]
['setosa' 'versicolor' 'virginica']


Split data into train/test (optional)

In [123]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# 70% training, 30% test

# stratify=y ensures each class is represented proportionally

Train a classifier (KNN)


Formula behind KNN

When predicting:

Compute distance between query and all points

Choose the K closest

Majority vote → classification

(Optional) Use weights = 1/distance

In [124]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

Make predictions

In [125]:
y_pred = knn.predict(X_test)


Compute confusion matrix

* Rows = actual labels

* Columns = predicted labels

In [126]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  2 13]]


| Class      | TP | FP | FN | TN |
| ---------- | -- | -- | -- | -- |
| setosa     | 16 | 0  | 0  | 29 |
| versicolor | 14 | 2  | 1  | 28 |
| virginica  | 12 | 1  | 2  | 30 |


In [127]:
print("\nClassification Report:\n", classification_report(
    y_test, y_pred, target_names=label_names))


Classification Report:
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.88      1.00      0.94        15
   virginica       1.00      0.87      0.93        15

    accuracy                           0.96        45
   macro avg       0.96      0.96      0.96        45
weighted avg       0.96      0.96      0.96        45



In [128]:
df_results = pd.DataFrame({
    "Actual": [label_names[i] for i in y_test],
    "Predicted": [label_names[i] for i in y_pred]
})
print(df_results.head(10))

       Actual   Predicted
0   virginica   virginica
1  versicolor  versicolor
2   virginica  versicolor
3  versicolor  versicolor
4   virginica   virginica
5   virginica   virginica
6  versicolor  versicolor
7  versicolor  versicolor
8      setosa      setosa
9   virginica   virginica


# Now, you try: Wine Dataset

In [129]:
from sklearn.datasets import load_wine
import numpy as np
import pandas as pd

wine = load_wine()
docs = wine.data        # numeric features, shape (178,13)
labels = wine.target    # 0,1,2
label_names = wine.target_names

print("Documents shape:", docs.shape)
print("Labels:", np.unique(labels))

Documents shape: (178, 13)
Labels: [0 1 2]


In [130]:
print(labels)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


Pick a query document to retrieve similar "documents"

In [132]:
query_idx = 0
query_vec = docs[query_idx].reshape(1, -1) # numeric features only
query_label = labels[query_idx]
print(query_label)
print("Query label:", label_names[query_idx])

0
Query label: class_0


Compute cosine similarity

In [133]:
from sklearn.metrics.pairwise import cosine_similarity

sims = cosine_similarity(query_vec, docs).flatten()

In [134]:
print(sims)

[1.       0.999709 0.99943  0.999085 0.999073 0.999118 0.999005 0.999668
 0.999657 0.999681 0.998773 0.998999 0.998672 0.999199 0.998571 0.999435
 0.999678 0.999844 0.99849  0.999831 0.99911  0.999859 0.999768 0.999673
 0.999939 0.999441 0.999154 0.998947 0.999974 0.999656 0.999181 0.998794
 0.999896 0.99992  0.999821 0.999916 0.999977 0.999538 0.999735 0.998814
 0.999596 0.99948  0.999642 0.999411 0.999987 0.99986  0.999729 0.999861
 0.999756 0.999444 0.999235 0.999005 0.999664 0.999369 0.999969 0.999872
 0.999993 0.999251 0.999387 0.998731 0.999534 0.994642 0.999438 0.995735
 0.985339 0.999605 0.99915  0.999107 0.999593 0.996037 0.999956 0.994881
 0.997235 0.999645 0.999944 0.994242 0.994744 0.99457  0.998142 0.994687
 0.981911 0.999921 0.999699 0.998491 0.997951 0.994807 0.997595 0.998813
 0.999837 0.999687 0.998477 0.996889 0.998704 0.983131 0.986683 0.998603
 0.995527 0.996637 0.999793 0.994916 0.999797 0.999126 0.994223 0.995726
 0.999835 0.985587 0.999009 0.998102 0.984643 0.999

Apply threshold for retrieval

In [135]:
threshold = 0.95  # choose a value between 0 and 1
retrieved_indices = np.where(sims >= threshold)[0]
print("Retrieved document indices:", retrieved_indices)

Retrieved document indices: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177]


Build y_true and y_pred

In [136]:
# Actual relevance: 1 if same class as query, else 0
y_true = [1 if labels[i] == query_label else 0 for i in range(len(labels))]

# Predicted relevance: 1 if retrieved, else 0
y_pred = [1 if sims[i] >= threshold else 0 for i in range(len(labels))]

Computer Confusion Matrix

|                         | Predicted 0 (Not Relevant) | Predicted 1 (Retrieved/Relevant) |
| ----------------------- | -------------------------- | -------------------------------- |
| Actual 0 (Not Relevant) | TN                         | FP                               |
| Actual 1 (Relevant)     | FN                         | TP                               |

In [137]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n", classification_report(
    y_true, y_pred, target_names=["Not Relevant", "Relevant"]))

Confusion Matrix:
 [[  0 119]
 [  0  59]]

Classification Report:
               precision    recall  f1-score   support

Not Relevant       0.00      0.00      0.00       119
    Relevant       0.33      1.00      0.50        59

    accuracy                           0.33       178
   macro avg       0.17      0.50      0.25       178
weighted avg       0.11      0.33      0.17       178



c:\Users\Adecarvalhoalvarezz\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Adecarvalhoalvarezz\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Adecarvalhoalvarezz\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier

In [138]:
print("Retrieved documents and similarities:")
for idx in retrieved_indices:
    print(f"Doc {idx} | similarity={sims[idx]:.4f} | label={label_names[labels[idx]]} | features={docs[idx]}")

Retrieved documents and similarities:
Doc 0 | similarity=1.0000 | label=class_0 | features=[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
 2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
Doc 1 | similarity=0.9997 | label=class_0 | features=[1.32e+01 1.78e+00 2.14e+00 1.12e+01 1.00e+02 2.65e+00 2.76e+00 2.60e-01
 1.28e+00 4.38e+00 1.05e+00 3.40e+00 1.05e+03]
Doc 2 | similarity=0.9994 | label=class_0 | features=[1.316e+01 2.360e+00 2.670e+00 1.860e+01 1.010e+02 2.800e+00 3.240e+00
 3.000e-01 2.810e+00 5.680e+00 1.030e+00 3.170e+00 1.185e+03]
Doc 3 | similarity=0.9991 | label=class_0 | features=[1.437e+01 1.950e+00 2.500e+00 1.680e+01 1.130e+02 3.850e+00 3.490e+00
 2.400e-01 2.180e+00 7.800e+00 8.600e-01 3.450e+00 1.480e+03]
Doc 4 | similarity=0.9991 | label=class_0 | features=[1.324e+01 2.590e+00 2.870e+00 2.100e+01 1.180e+02 2.800e+00 2.690e+00
 3.900e-01 1.820e+00 4.320e+00 1.040e+00 2.930e+00 7.350e+02]
Doc 5 | similarity=0.9991 | label=class_0 | f